# Create freshness registry
Defines which tables to check for freshness and the maximum age each table may reach before it's considered stale.

In [0]:
%sql
-- Freshness registry: which tables to check and how recently each must have changed
create or replace table validation_sweep.metadata.freshness_registry (
  catalog_name   string     not null,
  schema_name    string     not null,
  table_name     string     not null,
  max_age_hours  double     not null,
  is_active      boolean    not null,
  updated_at     timestamp  not null
);

-- Register a regularly updated table and a table that is never updated
insert into validation_sweep.metadata.freshness_registry values
  ('validation_sweep', 'dev', 'daily_orders', 24, true, current_timestamp()),
  ('validation_sweep', 'dev', 'customer',     24, true, current_timestamp());

In [0]:
# Verify every registered table exists and its threshold is positive; report all errors before failing
registry = spark.table("validation_sweep.metadata.freshness_registry").collect()
errors = []

for r in registry:
    full_name = f"{r.catalog_name}.{r.schema_name}.{r.table_name}"
    problems = []
    if not spark.catalog.tableExists(full_name):
        problems.append("table not found")
    if r.max_age_hours <= 0:
        problems.append(f"max_age_hours must be positive, found {r.max_age_hours}")

    if problems:
        errors.append(f"{full_name}: {', '.join(problems)}")
        print(f"{full_name:35} FAIL")
    else:
        print(f"{full_name:35} max {r.max_age_hours} hours  OK")

if errors:
    raise ValueError("\n".join(errors))